In [ ]:
import requests
import re
from bs4 import BeautifulSoup as bs
import pandas as pd
import datetime as dt
import json

import warnings
warnings.filterwarnings('ignore')


In [77]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
#lesum inn gögn af epl offical og understat
import pickle

pick_read = open('..data/understat_epl_ready.pickle','rb')
understat_epl_ready = pickle.load(pick_read)
pick_read.close()

pick_read = open('..data/understat_matches_2025.pickle','rb')
understat_epl_new = pickle.load(pick_read)
pick_read.close()

pick_read = open('..data/Epl_official_df_updated.pickle','rb')
Epl_official_ready = pickle.load(pick_read)
pick_read.close()

pick_read = open('..data/stats_2025-26.pickle','rb')
Epl_official_new = pickle.load(pick_read)
pick_read.close()

pick_read = open('..data/matches_2025-26.pickle','rb')
Epl_official_new_matches = pickle.load(pick_read)
pick_read.close()


In [79]:
import pandas as pd

understat_epl_new = pd.DataFrame({
    "Date": pd.to_datetime(understat_epl_new["date"]).dt.strftime("%Y/%m/%d"),
    "Home_Team": understat_epl_new["team_h"],
    "Away_Team": understat_epl_new["team_a"],
    "Home_Goals": pd.to_numeric(understat_epl_new["h_goals"], errors="coerce"),
    "Away_Goals": pd.to_numeric(understat_epl_new["a_goals"], errors="coerce"),
    "Home_xG": pd.to_numeric(understat_epl_new["h_xg"], errors="coerce"),
    "Away_xG": pd.to_numeric(understat_epl_new["a_xg"], errors="coerce"),
    "Home_Deep": pd.to_numeric(understat_epl_new["h_deep"], errors="coerce"),
    "Away_Deep": pd.to_numeric(understat_epl_new["a_deep"], errors="coerce"),
    "Home_ppda": pd.to_numeric(understat_epl_new["h_ppda"], errors="coerce"),
    "Away_ppda": pd.to_numeric(understat_epl_new["a_ppda"], errors="coerce"),
})

In [80]:
understat_full = pd.concat([understat_epl_ready, understat_epl_new], ignore_index=True).drop_duplicates(
    ["Date", "Home_Team", "Away_Team"])

In [81]:
Epl_official_new = Epl_official_new_matches.merge(Epl_official_new, on="url", how="inner")

In [82]:
Epl_official_ready = Epl_official_ready.rename(columns={"Match_Date": "Date"})

In [83]:
Epl_official_new = Epl_official_new.rename(columns={"Home_Through_Balls": "Home_Through_Balls_(%_Completed)",
                                                    "Away_Through_Balls": "Away_Through_Balls_(%_Completed)",})

In [84]:
Epl_official_new["Season"] = "25/26"
Epl_official_new["Matchweek"] = "Matchweek " + Epl_official_new["Matchweek"].astype(str)

In [85]:
pl_all = pd.concat([Epl_official_ready, Epl_official_new], ignore_index=True)
print(pl_all.shape)

(3420, 72)


In [86]:
pl_all = pl_all.drop(columns=["url", "Home_Passes", "Away_Passes"])

In [87]:
##dálkar með nan eftir concat, mismunur milli gagnasetta
nan_by_col = pl_all.isna().sum()

In [88]:
#til að skoða alla nan
#with pd.option_context('display.max_rows', None):
  #print(nan_by_col)
#sjáum villur í skraparanum.

In [89]:
cols_to_drop = pl_all.columns[pl_all.isna().sum() > 300].tolist()
pl_all = pl_all.drop(columns=cols_to_drop)

In [90]:
#skröpunarvillur í 11 leikjum
core_miss = pl_all[pl_all["Home_Possession"].isna() | pl_all["Home_Total_Shots"].isna()]
cols = ["Date", "Home_Team", "Away_Team", "Matchweek", "Season",
        "Home_Possession", "Home_Total_Shots", "Home_Total_Passes", "Home_Touches", "Home_Saves"]
print(core_miss[cols].to_string())
print("\ncount:", len(core_miss))

            Date       Home_Team    Away_Team     Matchweek Season Home_Possession Home_Total_Shots Home_Total_Passes Home_Touches Home_Saves
1144  2020/09/13           Spurs      Everton   Matchweek 1  20/21             NaN                9         552 (85%)          730          3
1240  2020/12/05        Man City       Fulham  Matchweek 11  20/21             NaN              NaN               NaN          NaN        NaN
1242  2020/12/06           Spurs      Arsenal  Matchweek 11  20/21             NaN              NaN               NaN          NaN        NaN
1243  2020/12/06       Liverpool       Wolves  Matchweek 11  20/21             NaN              NaN               NaN          NaN        NaN
1246  2020/12/07        Brighton  Southampton  Matchweek 11  20/21             NaN              NaN               NaN          NaN        NaN
1247  2020/12/11           Leeds     West Ham  Matchweek 12  20/21             NaN              NaN               NaN          NaN        NaN
1248  

In [91]:
#11 leikir þar sem villa er í skrapara. Sættum okkur við það.
#Fyllum því inn allar NAN með meðaltali liðs eftir að við erum búin að laga dálkana. Það er öruggast þar sem stundum vantar skráningar.


In [92]:
#lesum inn odds af football-data.co.uk
csvs = ['https://www.football-data.co.uk/mmz4281/1718/E0.csv',
        'https://www.football-data.co.uk/mmz4281/1819/E0.csv',
        'https://www.football-data.co.uk/mmz4281/1920/E0.csv',
        'https://www.football-data.co.uk/mmz4281/2021/E0.csv',
        'https://www.football-data.co.uk/mmz4281/2122/E0.csv',
        'https://www.football-data.co.uk/mmz4281/2223/E0.csv',
        'https://www.football-data.co.uk/mmz4281/2324/E0.csv',
        'https://www.football-data.co.uk/mmz4281/2425/E0.csv',
        'https://www.football-data.co.uk/mmz4281/2526/E0.csv']
odds = pd.DataFrame()
for c in csvs:
    odds = pd.concat([odds, pd.read_csv(c)])
odds.to_csv("football_odds.csv.gzip", index=False, compression='gzip') #vista saman
odds.shape

(3420, 180)

In [93]:
odds_df = odds[['Date','HomeTeam','AwayTeam', 'HTR','HS', 'AS', 'HST', 'AST', 'HC', 'AC',
                'HF','AF', 'B365H','B365D','B365A' ]]
odds_df.columns = ['Date','Home_Team','Away_Team', 'Half Time Result','HomeShots','AwayShots','HomeShotOT','AwayShotsOT',
                   'HomeFouls','AwayFouls','HomeCorners', 'AwayCorners','OddsH','OddsD','OddsA']
odds_df.head()

,Date,Home_Team,Away_Team,Half Time Result,HomeShots,AwayShots,HomeShotOT,AwayShotsOT,HomeFouls,AwayFouls,HomeCorners,AwayCorners,OddsH,OddsD,OddsA
0,11/08/2017,Arsenal,Leicester,D,27,6,10,3,9,4,9,12,1.53,4.5,6.50
1,12/08/2017,Brighton,Man City,D,6,14,2,4,3,10,6,9,11.00,5.5,1.33
2,12/08/2017,Chelsea,Burnley,A,19,10,6,5,8,5,16,11,1.25,6.5,15.00
3,12/08/2017,Crystal Palace,Huddersfield,A,14,8,4,6,12,9,7,19,1.83,3.6,5.00
4,12/08/2017,Everton,Stoke,H,9,9,4,1,6,7,13,10,1.70,3.8,5.75


In [94]:
#þurfum bara odds úr odds_df, lögum einnig date
odds_df.columns

Index(['Date', 'Home_Team', 'Away_Team', 'Half Time Result', 'HomeShots',
       'AwayShots', 'HomeShotOT', 'AwayShotsOT', 'HomeFouls', 'AwayFouls',
       'HomeCorners', 'AwayCorners', 'OddsH', 'OddsD', 'OddsA'],
      dtype='str')

In [95]:
odds_df = odds_df.drop(['Half Time Result', 'HomeShots',
       'AwayShots', 'HomeShotOT', 'AwayShotsOT', 'HomeFouls', 'AwayFouls',
       'HomeCorners', 'AwayCorners'], axis=1)
odds_df.head()

,Date,Home_Team,Away_Team,OddsH,OddsD,OddsA
0,11/08/2017,Arsenal,Leicester,1.53,4.5,6.50
1,12/08/2017,Brighton,Man City,11.00,5.5,1.33
2,12/08/2017,Chelsea,Burnley,1.25,6.5,15.00
3,12/08/2017,Crystal Palace,Huddersfield,1.83,3.6,5.00
4,12/08/2017,Everton,Stoke,1.70,3.8,5.75


In [96]:
odds_df['Date'] = pd.to_datetime(odds_df['Date'], format='%d/%m/%Y', errors='coerce')
odds_df['Date'] = odds_df['Date'].dt.strftime('%Y/%m/%d')
odds_df.head()

,Date,Home_Team,Away_Team,OddsH,OddsD,OddsA
0,2017/08/11,Arsenal,Leicester,1.53,4.5,6.50
1,2017/08/12,Brighton,Man City,11.00,5.5,1.33
2,2017/08/12,Chelsea,Burnley,1.25,6.5,15.00
3,2017/08/12,Crystal Palace,Huddersfield,1.83,3.6,5.00
4,2017/08/12,Everton,Stoke,1.70,3.8,5.75


In [97]:
#skoða hvort lið séu með mismunandi nöfn milli gagnasafna
a = (odds_df.Home_Team.unique())
b = (understat_full.Home_Team.unique())
c = (pl_all.Home_Team.unique())
print(sorted(a))
print(sorted(b))
print(sorted(c))

['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Cardiff', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Huddersfield', 'Ipswich', 'Leeds', 'Leicester', 'Liverpool', 'Luton', 'Man City', 'Man United', 'Newcastle', 'Norwich', "Nott'm Forest", 'Sheffield United', 'Southampton', 'Stoke', 'Sunderland', 'Swansea', 'Tottenham', 'Watford', 'West Brom', 'West Ham', 'Wolves']
['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Cardiff', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Huddersfield', 'Ipswich', 'Leeds', 'Leicester', 'Liverpool', 'Luton', 'Manchester City', 'Manchester United', 'Newcastle United', 'Norwich', 'Nottingham Forest', 'Sheffield United', 'Southampton', 'Stoke', 'Sunderland', 'Swansea', 'Tottenham', 'Watford', 'West Bromwich Albion', 'West Ham', 'Wolverhampton Wanderers']
['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Brighton and Hove Albion', 'Burnley', 'Cardiff', 'Chelsea', 'Crystal 

In [98]:
replace_values = {'Man City' : 'Manchester City', 'Man United' : 'Manchester United', 'Man Utd' : 'Manchester United', 'Newcastle' : 'Newcastle United',
                  'Nott\'m Forest': 'Nottingham Forest', 'West Brom' : 'West Bromwich Albion', 'Sheffield Utd' : 'Sheffield United',
                  'Sheff Utd' : 'Sheffield United', 'Spurs' : 'Tottenham', 'Tottenham Hotspur' : 'Tottenham', 'Wolves' : 'Wolverhampton Wanderers',
                  'Brighton and Hove Albion' : 'Brighton', 'Leeds United' : 'Leeds', 'West Ham United' : 'West Ham'}

odds_df['Home_Team'] = odds_df['Home_Team'].map(replace_values).fillna(odds_df['Home_Team'])
odds_df['Away_Team'] = odds_df['Away_Team'].map(replace_values).fillna(odds_df['Away_Team'])
pl_all['Home_Team'] = pl_all['Home_Team'].map(replace_values).fillna(pl_all['Home_Team'])
pl_all['Away_Team'] = pl_all['Away_Team'].map(replace_values).fillna(pl_all['Away_Team'])

In [99]:
#breyta í prósentur
for x in ['OddsH','OddsD','OddsA']:
    odds_df[x]=1/odds_df[x]

odds_df.shape

(3420, 6)

In [100]:
#mergea saman

test = pd.merge(left=understat_full, right=odds_df,
                  left_on= ['Date','Home_Team', 'Away_Team'],
                   right_on=['Date','Home_Team', 'Away_Team'],
                  how='left')




In [101]:
Epl_data_ready = pd.merge(left=test, right=pl_all,
                  left_on= ['Date','Home_Team', 'Away_Team'],
                   right_on=['Date','Home_Team', 'Away_Team'],
                  how='left')

In [102]:
Epl_data_ready.shape

(3420, 58)

In [103]:
#brjótum upp dálka sem eru með %completed eða slíkt
def split_stat_column(df, column_name):
    base_name = column_name.replace('_(%_Completed)', '')

    #Skipta upp dálkagildum
    extracted = df[column_name].str.extract(r'(\d+)\s*\((\d+)%\)')

    #breytum % í float
    df[base_name] = pd.to_numeric(extracted[0], errors='coerce')
    df[f"{base_name}_Completed"] = pd.to_numeric(extracted[1], errors='coerce') / 100

    #henda gamla dálknum
    df.drop(columns=[column_name], inplace=True)


In [104]:
pd.set_option('display.max_columns', None)
Epl_data_ready = Epl_data_ready.rename(columns={"Home_Total_Passes": "Home_Total_Passes_(%_Completed)",
                                                    "Away_Total_Passes": "Away_Total_Passes_(%_Completed)",})
Epl_data_ready.head()

,Date,Home_Team,Away_Team,Home_Goals,Away_Goals,Home_xG,Away_xG,Home_Deep,Away_Deep,Home_ppda,Away_ppda,OddsH,OddsD,OddsA,Matchweek,Home_Possession,Away_Possession,Home_Total_Shots,Away_Total_Shots,Home_Shots_On_Target,Away_Shots_On_Target,Home_Corners,Away_Corners,Home_Saves,Away_Saves,Home_Shots_Off_Target,Away_Shots_Off_Target,Home_Shots_Inside_the_Box,Away_Shots_Inside_the_Box,Home_Shots_Outside_the_Box,Away_Shots_Outside_the_Box,Home_Big_Chances_Created,Away_Big_Chances_Created,Home_Total_Crosses_(%_Completed),Away_Total_Crosses_(%_Completed),Home_Total_Passes_(%_Completed),Away_Total_Passes_(%_Completed),Home_Long_Passes_(%_Completed),Away_Long_Passes_(%_Completed),Home_Through_Balls_(%_Completed),Away_Through_Balls_(%_Completed),Home_Touches,Away_Touches,Home_Touches_in_the_opposition_box,Away_Touches_in_the_opposition_box,Home_Tackles_Won,Away_Tackles_Won,Home_Blocks,Away_Blocks,Home_Interceptions,Away_Interceptions,Home_Clearances,Away_Clearances,Home_Duels_Won,Away_Duels_Won,Home_Aerial_Duels_Won,Away_Aerial_Duels_Won,Season
0,2017/08/11,Arsenal,Leicester,4,3,2.54329,1.46495,13,2,5.4444,13.5455,0.653595,0.222222,0.153846,Matchweek 1,70%,30%,27,6,10,3,9,4,0,6,9,3,3,0,12,1,4,2,20 (30%),18 (22%),632 (85%),263 (63%),55 (67%),70 (46%),2 (50%),2 (50%),859,459,30,11,17 (74%),8 (47%),9,8,13,11,30,24,69,48,18,18,17/18
1,2017/08/12,Brighton,Manchester City,0,2,0.276343,1.86751,2,18,67.3333,5.4444,0.090909,0.181818,0.751880,Matchweek 1,21.8%,78.2%,6,14,2,4,3,10,3,2,2,5,2,1,2,8,0,3,8 (25%),27 (15%),213 (61%),768 (90%),49 (37%),52 (65%),0 (0%),3 (67%),375,918,9,28,8 (80%),6 (60%),7,7,12,9,39,8,38,35,13,13,17/18
2,2017/08/12,Watford,Liverpool,3,3,2.17647,2.61549,2,10,11.2609,14.7143,0.166667,0.238095,0.617284,Matchweek 1,45.6%,54.4%,9,14,4,5,3,3,2,1,4,8,0,1,3,1,1,2,16 (13%),14 (29%),395 (70%),477 (74%),84 (42%),71 (44%),2 (50%),2 (0%),595,682,11,24,12 (63%),15 (75%),2,2,11,11,28,22,60,64,22,25,17/18
3,2017/08/12,West Bromwich Albion,Bournemouth,1,0,1.18399,0.378659,6,6,17.3889,12.0000,0.416667,0.303030,0.303030,Matchweek 1,28.7%,71.3%,16,9,6,2,8,2,2,5,9,2,3,1,5,4,1,0,16 (38%),24 (21%),242 (65%),612 (86%),76 (34%),68 (37%),2 (100%),1 (0%),419,775,19,19,15 (79%),13 (81%),6,6,14,5,26,17,34,52,12,18,17/18
4,2017/08/12,Southampton,Swansea,0,0,2.21748,0.406196,16,1,8.5200,12.8824,0.617284,0.250000,0.153846,Matchweek 1,59.6%,40.4%,29,4,2,0,13,0,0,2,16,2,NaN,NaN,13,1,2,0,35 (34%),8 (25%),518 (83%),365 (78%),46 (39%),71 (49%),1 (0%),0 (0%),707,532,38,7,10 (83%),8 (73%),15,13,12,8,11,38,47,42,17,18,17/18


In [105]:
cols_to_split = [
    'Home_Total_Crosses_(%_Completed)',
    'Away_Total_Crosses_(%_Completed)',
    'Home_Total_Passes_(%_Completed)',
    'Away_Total_Passes_(%_Completed)',
    'Home_Long_Passes_(%_Completed)',
    'Away_Long_Passes_(%_Completed)',
    'Home_Through_Balls_(%_Completed)',
    'Away_Through_Balls_(%_Completed)',
    'Home_Tackles_Won',
    'Away_Tackles_Won',
]

for col in cols_to_split:
    split_stat_column(Epl_data_ready, col)

In [106]:
#prósenta burt úr possession
cols = ['Home_Possession', 'Away_Possession']
for col in cols:
    Epl_data_ready[col] = pd.to_numeric(Epl_data_ready[col].str.rstrip('%'), errors='coerce') / 100


In [107]:
Epl_data_ready.head()

,Date,Home_Team,Away_Team,Home_Goals,Away_Goals,Home_xG,Away_xG,Home_Deep,Away_Deep,Home_ppda,Away_ppda,OddsH,OddsD,OddsA,Matchweek,Home_Possession,Away_Possession,Home_Total_Shots,Away_Total_Shots,Home_Shots_On_Target,Away_Shots_On_Target,Home_Corners,Away_Corners,Home_Saves,Away_Saves,Home_Shots_Off_Target,Away_Shots_Off_Target,Home_Shots_Inside_the_Box,Away_Shots_Inside_the_Box,Home_Shots_Outside_the_Box,Away_Shots_Outside_the_Box,Home_Big_Chances_Created,Away_Big_Chances_Created,Home_Touches,Away_Touches,Home_Touches_in_the_opposition_box,Away_Touches_in_the_opposition_box,Home_Blocks,Away_Blocks,Home_Interceptions,Away_Interceptions,Home_Clearances,Away_Clearances,Home_Duels_Won,Away_Duels_Won,Home_Aerial_Duels_Won,Away_Aerial_Duels_Won,Season,Home_Total_Crosses,Home_Total_Crosses_Completed,Away_Total_Crosses,Away_Total_Crosses_Completed,Home_Total_Passes,Home_Total_Passes_Completed,Away_Total_Passes,Away_Total_Passes_Completed,Home_Long_Passes,Home_Long_Passes_Completed,Away_Long_Passes,Away_Long_Passes_Completed,Home_Through_Balls,Home_Through_Balls_Completed,Away_Through_Balls,Away_Through_Balls_Completed,Home_Tackles_Won_Completed,Away_Tackles_Won_Completed
0,2017/08/11,Arsenal,Leicester,4,3,2.54329,1.46495,13,2,5.4444,13.5455,0.653595,0.222222,0.153846,Matchweek 1,0.700,0.300,27,6,10,3,9,4,0,6,9,3,3,0,12,1,4,2,859,459,30,11,9,8,13,11,30,24,69,48,18,18,17/18,20.0,0.30,18.0,0.22,632.0,0.85,263.0,0.63,55.0,0.67,70.0,0.46,2.0,0.5,2.0,0.50,0.74,0.47
1,2017/08/12,Brighton,Manchester City,0,2,0.276343,1.86751,2,18,67.3333,5.4444,0.090909,0.181818,0.751880,Matchweek 1,0.218,0.782,6,14,2,4,3,10,3,2,2,5,2,1,2,8,0,3,375,918,9,28,7,7,12,9,39,8,38,35,13,13,17/18,8.0,0.25,27.0,0.15,213.0,0.61,768.0,0.90,49.0,0.37,52.0,0.65,0.0,0.0,3.0,0.67,0.80,0.60
2,2017/08/12,Watford,Liverpool,3,3,2.17647,2.61549,2,10,11.2609,14.7143,0.166667,0.238095,0.617284,Matchweek 1,0.456,0.544,9,14,4,5,3,3,2,1,4,8,0,1,3,1,1,2,595,682,11,24,2,2,11,11,28,22,60,64,22,25,17/18,16.0,0.13,14.0,0.29,395.0,0.70,477.0,0.74,84.0,0.42,71.0,0.44,2.0,0.5,2.0,0.00,0.63,0.75
3,2017/08/12,West Bromwich Albion,Bournemouth,1,0,1.18399,0.378659,6,6,17.3889,12.0000,0.416667,0.303030,0.303030,Matchweek 1,0.287,0.713,16,9,6,2,8,2,2,5,9,2,3,1,5,4,1,0,419,775,19,19,6,6,14,5,26,17,34,52,12,18,17/18,16.0,0.38,24.0,0.21,242.0,0.65,612.0,0.86,76.0,0.34,68.0,0.37,2.0,1.0,1.0,0.00,0.79,0.81
4,2017/08/12,Southampton,Swansea,0,0,2.21748,0.406196,16,1,8.5200,12.8824,0.617284,0.250000,0.153846,Matchweek 1,0.596,0.404,29,4,2,0,13,0,0,2,16,2,NaN,NaN,13,1,2,0,707,532,38,7,15,13,12,8,11,38,47,42,17,18,17/18,35.0,0.34,8.0,0.25,518.0,0.83,365.0,0.78,46.0,0.39,71.0,0.49,1.0,0.0,0.0,0.00,0.83,0.73


In [108]:
#nú getum við lagað NAN leikina og skipt inn meðaltali liðs.
#lögum fyrst object breytur
#print(Epl_data_ready.dtypes[Epl_data_ready.dtypes == object])
id_cols = ['Date', 'Home_Team', 'Away_Team', 'Matchweek', 'Season']
stat_cols = [c for c in Epl_data_ready.columns if c not in id_cols]
Epl_data_ready[stat_cols] = Epl_data_ready[stat_cols].apply(pd.to_numeric, errors='coerce')

In [109]:
for col in Epl_data_ready.columns:
    if Epl_data_ready[col].isna().any():
        if col.startswith('Home_'):
            Epl_data_ready[col] = Epl_data_ready[col].fillna(Epl_data_ready.groupby('Home_Team')[col].transform('mean'))
        elif col.startswith('Away_'):
            Epl_data_ready[col] = Epl_data_ready[col].fillna(Epl_data_ready.groupby('Away_Team')[col].transform('mean'))
print("remaining NaNs:", Epl_data_ready.isna().sum().sum())

remaining NaNs: 0


In [110]:
#cleaning done

In [111]:
#nú má framkvæma feature engineering og reikna út lýsandi gögn fyrir ml líkönin
#gaman væri að setja inn einhverskonar rating system svipað elo

In [112]:
#hvaða lið vann? Heima, draw eða útilið
import numpy as np

def match_winner(winner):
    winner['target'] = np.select(
        [winner['Home_Goals'] > winner['Away_Goals'],
         winner['Home_Goals'] == winner['Away_Goals']],
        ['W', 'D'],
        default='L'
    )
    return winner

In [113]:
Epl_data_ready = match_winner(Epl_data_ready)

In [114]:
Epl_data_ready = match_winner(Epl_data_ready)
print(Epl_data_ready['target'].value_counts(normalize=True))

target
W    0.441228
L    0.325146
D    0.233626
Name: proportion, dtype: float64


In [115]:
Epl_data_ready['GW'] = Epl_data_ready['Matchweek'].str.replace('Matchweek ', '', regex=False).astype(int)

In [116]:
import numpy as np

def get_agg_points(df):
    df = df.copy()
    df['_row'] = range(len(df))

    df['_pts_home'] = np.select([df['target']=='W', df['target']=='D'], [3,1], 0)
    df['_pts_away'] = np.select([df['target']=='L', df['target']=='D'], [3,1], 0)

    home = df[['_row','Season','GW','Home_Team','_pts_home']].rename(
        columns={'Home_Team':'Team','_pts_home':'pts'}); home['side']='H'
    away = df[['_row','Season','GW','Away_Team','_pts_away']].rename(
        columns={'Away_Team':'Team','_pts_away':'pts'}); away['side']='A'

    long = pd.concat([home, away]).sort_values(['Season','Team','GW'])
    ## stig fyrir spilaðan leik, eitt lið per línu.
    long['cum_before'] = long.groupby(['Season','Team'])['pts'].cumsum() - long['pts']

    htp = long[long.side=='H'].set_index('_row')['cum_before']
    atp = long[long.side=='A'].set_index('_row')['cum_before']
    df['HTP'] = df['_row'].map(htp)
    df['ATP'] = df['_row'].map(atp)
    return df.drop(columns=['_row','_pts_home','_pts_away'])

Epl_data_ready = get_agg_points(Epl_data_ready)

In [117]:
Epl_data_ready.tail(10)

,Date,Home_Team,Away_Team,Home_Goals,Away_Goals,Home_xG,Away_xG,Home_Deep,Away_Deep,Home_ppda,Away_ppda,OddsH,OddsD,OddsA,Matchweek,Home_Possession,Away_Possession,Home_Total_Shots,Away_Total_Shots,Home_Shots_On_Target,Away_Shots_On_Target,Home_Corners,Away_Corners,Home_Saves,Away_Saves,Home_Shots_Off_Target,Away_Shots_Off_Target,Home_Shots_Inside_the_Box,Away_Shots_Inside_the_Box,Home_Shots_Outside_the_Box,Away_Shots_Outside_the_Box,Home_Big_Chances_Created,Away_Big_Chances_Created,Home_Touches,Away_Touches,Home_Touches_in_the_opposition_box,Away_Touches_in_the_opposition_box,Home_Blocks,Away_Blocks,Home_Interceptions,Away_Interceptions,Home_Clearances,Away_Clearances,Home_Duels_Won,Away_Duels_Won,Home_Aerial_Duels_Won,Away_Aerial_Duels_Won,Season,Home_Total_Crosses,Home_Total_Crosses_Completed,Away_Total_Crosses,Away_Total_Crosses_Completed,Home_Total_Passes,Home_Total_Passes_Completed,Away_Total_Passes,Away_Total_Passes_Completed,Home_Long_Passes,Home_Long_Passes_Completed,Away_Long_Passes,Away_Long_Passes_Completed,Home_Through_Balls,Home_Through_Balls_Completed,Away_Through_Balls,Away_Through_Balls_Completed,Home_Tackles_Won_Completed,Away_Tackles_Won_Completed,target,GW,HTP,ATP
3410,2026/05/24,Brighton,Manchester United,0,3,0.702398,2.328680,10,9,7.6129,15.7059,0.540541,0.238095,0.277778,Matchweek 38,0.524,0.476,13.0,11.0,2.0,7.0,0.0,3.0,5.0,2.0,6.0,3.0,2.0,3.0,4.0,4.0,0.0,4.0,658.0,604.0,32.0,32.0,6.0,4.0,8.0,10.0,12.0,14.0,40.0,40.0,13.0,4.0,25/26,20.0,0.20,11.0,0.09,484.0,0.86,450.0,0.83,37.0,0.49,55.0,0.40,0.0,0.0,1.0,1.0,0.71,0.67,L,38,53,68
3411,2026/05/24,Burnley,Wolverhampton Wanderers,1,1,1.284760,2.843300,13,5,9.3571,13.6667,0.425532,0.285714,0.344828,Matchweek 38,0.704,0.296,16.0,16.0,8.0,4.0,7.0,7.0,3.0,7.0,2.0,5.0,3.0,2.0,9.0,6.0,2.0,3.0,751.0,423.0,33.0,26.0,13.0,15.0,7.0,9.0,13.0,28.0,40.0,42.0,13.0,11.0,25/26,28.0,0.21,22.0,0.36,569.0,0.88,225.0,0.72,53.0,0.58,45.0,0.29,2.0,0.5,1.0,1.0,0.89,0.59,D,38,21,19
3412,2026/05/24,Crystal Palace,Arsenal,1,2,1.004920,3.906450,3,8,17.7500,9.8889,0.238095,0.250000,0.571429,Matchweek 38,0.397,0.603,8.0,17.0,3.0,7.0,3.0,4.0,5.0,2.0,3.0,6.0,2.0,4.0,0.0,3.0,3.0,6.0,491.0,688.0,21.0,41.0,6.0,6.0,10.0,5.0,23.0,28.0,42.0,40.0,11.0,14.0,25/26,18.0,0.22,13.0,0.38,328.0,0.80,518.0,0.89,47.0,0.38,36.0,0.39,0.0,0.0,6.0,0.5,0.33,0.43,L,38,45,82
3413,2026/05/24,Fulham,Newcastle United,2,0,1.728780,0.242320,10,6,11.4444,12.7647,0.347222,0.263158,0.444444,Matchweek 38,0.449,0.551,21.0,7.0,6.0,2.0,6.0,6.0,2.0,4.0,9.0,3.0,2.0,1.0,11.0,3.0,1.0,0.0,579.0,682.0,33.0,19.0,8.0,8.0,12.0,10.0,13.0,19.0,43.0,45.0,12.0,16.0,25/26,13.0,0.15,16.0,0.13,418.0,0.83,508.0,0.88,46.0,0.43,45.0,0.40,1.0,1.0,2.0,0.5,0.81,0.79,W,38,49,49
3414,2026/05/24,Liverpool,Brentford,1,1,2.994120,1.460820,11,5,11.7059,12.2222,0.555556,0.238095,0.263158,Matchweek 38,0.602,0.398,24.0,11.0,8.0,2.0,14.0,2.0,1.0,7.0,8.0,6.0,6.0,1.0,7.0,2.0,4.0,2.0,695.0,523.0,44.0,24.0,12.0,11.0,6.0,3.0,17.0,28.0,46.0,49.0,15.0,12.0,25/26,27.0,0.19,18.0,0.33,508.0,0.86,333.0,0.79,55.0,0.45,51.0,0.29,1.0,1.0,0.0,0.0,0.27,0.48,D,38,59,52
3415,2026/05/24,Manchester City,Aston Villa,1,2,1.380850,2.439700,14,4,16.2000,20.7000,0.775194,0.166667,0.111111,Matchweek 38,0.525,0.475,16.0,12.0,3.0,5.0,9.0,4.0,3.0,2.0,7.0,5.0,2.0,1.0,6.0,3.0,1.0,3.0,635.0,616.0,46.0,20.0,9.0,9.0,7.0,11.0,15.0,37.0,26.0,42.0,7.0,3.0,25/26,14.0,0.07,8.0,0.38,481.0,0.89,440.0,0.90,30.0,0.63,40.0,0.50,4.0,0.5,1.0,1.0,0.60,0.68,L,38,78,62
3416,2026/05/24,Nottingham Forest,Bournemouth,1,1,2.058520,1.336840,3,8,17.9167,8.0000,0.303030,0.266667,0.487805,Matchweek 38,0.452,0.548,15.0,17.0,5.0,4.0,6.0,3.0,3.0,3.0,5.0,6.0,4.0,2.0,5.0,11.0,2.0,1.0,579.0,697.0,24.0,31.0,13.0,12.0,10.0,18.0,28.0,18.0,41.0,54.0,14.0,12.0,25/26,17.0,0.35,15.0,0.13,396.0,0.78,485.0,0.84,52.0,0.35,50.0,0.42,2.0,0.0,0.0,0.0,0.50,0.38,D,38,43,56
3417,2026/05/24,Sunderland,Chelsea,2,1,2.331090,0.920885,12,7,7.9375,17.2727,0.277778,0.277778,0.500000,Matchweek 38,0.439,0.561,21.0,8.0,6.0,3.0,6.0,2.0,2.0,5

In [118]:
def add_form(df, n=5):
    """Rolling last-n form per team per season, BEFORE current match.
    Adds HTForm/ATForm (string like 'WWDLW') and HTFormPts/ATFormPts."""
    df = df.copy()
    df['_row'] = range(len(df))
    df['_pts_home'] = np.select([df['target']=='W', df['target']=='D'], [3,1], 0)
    df['_pts_away'] = np.select([df['target']=='L', df['target']=='D'], [3,1], 0)
    df['_res_home'] = df['target']
    df['_res_away'] = df['target'].map({'W':'L','L':'W','D':'D'})

    home = df[['_row','Season','GW','Home_Team','_pts_home','_res_home']].rename(
        columns={'Home_Team':'Team','_pts_home':'pts','_res_home':'res'}); home['side']='H'
    away = df[['_row','Season','GW','Away_Team','_pts_away','_res_away']].rename(
        columns={'Away_Team':'Team','_pts_away':'pts','_res_away':'res'}); away['side']='A'
    long = pd.concat([home, away]).sort_values(['Season','Team','GW'])

    g = long.groupby(['Season','Team'])
    long['form_pts'] = (g['pts'].apply(lambda s: s.shift(1).rolling(n, min_periods=1).sum())
                                .reset_index(level=[0,1], drop=True))
    def form_str(s):
        out, hist = [], []
        for r in s:
            out.append(''.join(hist[-n:]))
            hist.append(r)
        return pd.Series(out, index=s.index)
    long['form_str'] = g['res'].apply(form_str).reset_index(level=[0,1], drop=True)

    for side, hp, sp in [('H','HTFormPts','HTForm'), ('A','ATFormPts','ATForm')]:
        sub = long[long.side==side].set_index('_row')
        df[hp] = df['_row'].map(sub['form_pts']).fillna(0)
        df[sp] = df['_row'].map(sub['form_str']).fillna('')
    return df.drop(columns=['_row','_pts_home','_pts_away','_res_home','_res_away'])

Epl_data_ready = add_form(Epl_data_ready, n=5)

In [119]:
Epl_data_ready.tail(10)

,Date,Home_Team,Away_Team,Home_Goals,Away_Goals,Home_xG,Away_xG,Home_Deep,Away_Deep,Home_ppda,Away_ppda,OddsH,OddsD,OddsA,Matchweek,Home_Possession,Away_Possession,Home_Total_Shots,Away_Total_Shots,Home_Shots_On_Target,Away_Shots_On_Target,Home_Corners,Away_Corners,Home_Saves,Away_Saves,Home_Shots_Off_Target,Away_Shots_Off_Target,Home_Shots_Inside_the_Box,Away_Shots_Inside_the_Box,Home_Shots_Outside_the_Box,Away_Shots_Outside_the_Box,Home_Big_Chances_Created,Away_Big_Chances_Created,Home_Touches,Away_Touches,Home_Touches_in_the_opposition_box,Away_Touches_in_the_opposition_box,Home_Blocks,Away_Blocks,Home_Interceptions,Away_Interceptions,Home_Clearances,Away_Clearances,Home_Duels_Won,Away_Duels_Won,Home_Aerial_Duels_Won,Away_Aerial_Duels_Won,Season,Home_Total_Crosses,Home_Total_Crosses_Completed,Away_Total_Crosses,Away_Total_Crosses_Completed,Home_Total_Passes,Home_Total_Passes_Completed,Away_Total_Passes,Away_Total_Passes_Completed,Home_Long_Passes,Home_Long_Passes_Completed,Away_Long_Passes,Away_Long_Passes_Completed,Home_Through_Balls,Home_Through_Balls_Completed,Away_Through_Balls,Away_Through_Balls_Completed,Home_Tackles_Won_Completed,Away_Tackles_Won_Completed,target,GW,HTP,ATP,HTFormPts,HTForm,ATFormPts,ATForm
3410,2026/05/24,Brighton,Manchester United,0,3,0.702398,2.328680,10,9,7.6129,15.7059,0.540541,0.238095,0.277778,Matchweek 38,0.524,0.476,13.0,11.0,2.0,7.0,0.0,3.0,5.0,2.0,6.0,3.0,2.0,3.0,4.0,4.0,0.0,4.0,658.0,604.0,32.0,32.0,6.0,4.0,8.0,10.0,12.0,14.0,40.0,40.0,13.0,4.0,25/26,20.0,0.20,11.0,0.09,484.0,0.86,450.0,0.83,37.0,0.49,55.0,0.40,0.0,0.0,1.0,1.0,0.71,0.67,L,38,53,68,7.0,WDLWL,13.0,WWWDW
3411,2026/05/24,Burnley,Wolverhampton Wanderers,1,1,1.284760,2.843300,13,5,9.3571,13.6667,0.425532,0.285714,0.344828,Matchweek 38,0.704,0.296,16.0,16.0,8.0,4.0,7.0,7.0,3.0,7.0,2.0,5.0,3.0,2.0,9.0,6.0,2.0,3.0,751.0,423.0,33.0,26.0,13.0,15.0,7.0,9.0,13.0,28.0,40.0,42.0,13.0,11.0,25/26,28.0,0.21,22.0,0.36,569.0,0.88,225.0,0.72,53.0,0.58,45.0,0.29,2.0,0.5,1.0,1.0,0.89,0.59,D,38,21,19,1.0,LLLDL,2.0,LLDLD
3412,2026/05/24,Crystal Palace,Arsenal,1,2,1.004920,3.906450,3,8,17.7500,9.8889,0.238095,0.250000,0.571429,Matchweek 38,0.397,0.603,8.0,17.0,3.0,7.0,3.0,4.0,5.0,2.0,3.0,6.0,2.0,4.0,0.0,3.0,3.0,6.0,491.0,688.0,21.0,41.0,6.0,6.0,10.0,5.0,23.0,28.0,42.0,40.0,11.0,14.0,25/26,18.0,0.22,13.0,0.38,328.0,0.80,518.0,0.89,47.0,0.38,36.0,0.39,0.0,0.0,6.0,0.5,0.33,0.43,L,38,45,82,2.0,LLDLD,12.0,LWWWW
3413,2026/05/24,Fulham,Newcastle United,2,0,1.728780,0.242320,10,6,11.4444,12.7647,0.347222,0.263158,0.444444,Matchweek 38,0.449,0.551,21.0,7.0,6.0,2.0,6.0,6.0,2.0,4.0,9.0,3.0,2.0,1.0,11.0,3.0,1.0,0.0,579.0,682.0,33.0,19.0,8.0,8.0,12.0,10.0,13.0,19.0,43.0,45.0,12.0,16.0,25/26,13.0,0.15,16.0,0.13,418.0,0.83,508.0,0.88,46.0,0.43,45.0,0.40,1.0,1.0,2.0,0.5,0.81,0.79,W,38,49,49,5.0,DWLLD,7.0,LLWDW
3414,2026/05/24,Liverpool,Brentford,1,1,2.994120,1.460820,11,5,11.7059,12.2222,0.555556,0.238095,0.263158,Matchweek 38,0.602,0.398,24.0,11.0,8.0,2.0,14.0,2.0,1.0,7.0,8.0,6.0,6.0,1.0,7.0,2.0,4.0,2.0,695.0,523.0,44.0,24.0,12.0,11.0,6.0,3.0,17.0,28.0,46.0,49.0,15.0,12.0,25/26,27.0,0.19,18.0,0.33,508.0,0.86,333.0,0.79,55.0,0.45,51.0,0.29,1.0,1.0,0.0,0.0,0.27,0.48,D,38,59,52,7.0,WWLDL,5.0,DLWLD
3415,2026/05/24,Manchester City,Aston Villa,1,2,1.380850,2.439700,14,4,16.2000,20.7000,0.775194,0.166667,0.111111,Matchweek 38,0.525,0.475,16.0,12.0,3.0,5.0,9.0,4.0,3.0,2.0,7.0,5.0,2.0,1.0,6.0,3.0,1.0,3.0,635.0,616.0,46.0,20.0,9.0,9.0,7.0,11.0,15.0,37.0,26.0,42.0,7.0,3.0,25/26,14.0,0.07,8.0,0.38,481.0,0.89,440.0,0.90,30.0,0.63,40.0,0.50,4.0,0.5,1.0,1.0,0.60,0.68,L,38,78,62,11.0,WDWWD,7.0,WLLDW
3416,2026/05/24,Nottingham Forest,Bournemouth,1,1,2.058520,1.336840,3,8,17.9167,8.0000,0.303030,0.266667,0.487805,Matchweek 38,0.452,0.548,15.0,17.0,5.0,4.0,6.0,3.0,3.0,3.0,5.0,6.0,4.0,2.0,5.0,11.0,2.0,1.0,579.0,697.0,24.0,31.0,13.0,12.0,10.0,18.0,28.0,18.0,41.0,54.0,14.0,12.0,25/26,17.0,0.35,15.0,0.13,396.0,0.78,485.0,0.84,52.0,0.35,50.0,0.42,2.0,0.0,0.0,0.0,0.50,0.38,D,38,43,56,10

In [120]:
#gameid
Epl_data_ready.insert(0, 'GameId', range(1, len(Epl_data_ready)+1))

In [121]:
#nú glicko
#!pip install glicko2
from glicko2 import Player

def add_glicko(df):
    """Glicko2 rating for each team going INTO each match (before it's played).
    One continuous timeline per team — rating reflects their most recent prior
    match, home or away. Adds Home_Rating / Away_Rating. Leakage-safe."""
    df = df.sort_values(['Season', 'GW', 'GameId']).reset_index(drop=True)

    teams = pd.concat([df['Home_Team'], df['Away_Team']]).unique()
    players = {t: Player() for t in teams}

    home_r, away_r = [], []
    for _, row in df.iterrows():
        h, a, res = row['Home_Team'], row['Away_Team'], row['target']

        # record rating BEFORE the update — this is what the match "sees"
        home_r.append(players[h].getRating())
        away_r.append(players[a].getRating())

        # then update both teams with this match's result
        hs = {'W': 1.0, 'D': 0.5, 'L': 0.0}[res]
        rh, rdh = players[h].getRating(), players[h].getRd()
        ra, rda = players[a].getRating(), players[a].getRd()
        players[h].update_player([ra], [rda], [hs])
        players[a].update_player([rh], [rdh], [1 - hs])

    df['Home_Rating'] = home_r
    df['Away_Rating'] = away_r
    return df

Epl_data_ready = add_glicko(Epl_data_ready)

Hér vil ég gera smá breytingar á framsetningu. Laga til breytur með því að setja fram "exponentially moving average" gildi á þeim. Form breyta, hversu vel er lið að standa sig í seinustu 5 leikjum er komin. Passa svo alltaf að spá ekki fyrir núverandi leik, nota shift.

In [125]:
exclude_stats = ['Rating', 'Team']

home_stats = [c for c in Epl_data_ready.columns
              if c.startswith('Home_')
              and c.replace('Home_','') not in exclude_stats]

In [131]:
def add_rolling_stats(df, home_stats, span=5):
    """Non-resetting EMA of each stat per team, BEFORE current match
    (shift(1) leakage guard). Carries across seasons; EMA decay fades old form.
    Adds Home_<stat>_ema / Away_<stat>_ema."""
    df = df.sort_values('GameId').reset_index(drop=True)
    df['_row'] = range(len(df))

    stats = [c.replace('Home_','') for c in home_stats]
    home = df[['_row','GameId','Home_Team'] + [f'Home_{s}' for s in stats]].copy()
    home.columns = ['_row','GameId','Team'] + stats; home['side'] = 'H'
    away = df[['_row','GameId','Away_Team'] + [f'Away_{s}' for s in stats]].copy()
    away.columns = ['_row','GameId','Team'] + stats; away['side'] = 'A'
    long = pd.concat([home, away]).sort_values(['Team','GameId'])

    for s in stats:
        long[s] = (long.groupby('Team')[s]
                       .apply(lambda x: x.shift(1).ewm(span=span, min_periods=1).mean())
                       .reset_index(level=0, drop=True))

    for side, pre in [('H','Home'), ('A','Away')]:
        sub = long[long.side==side].set_index('_row')
        for s in stats:
            df[f'{pre}_{s}_ema'] = df['_row'].map(sub[s])
    return df.drop(columns='_row')

Epl_data_ready = add_rolling_stats(Epl_data_ready, home_stats, span=5)

In [133]:
#skoðum hvernig þetta gekk
ema_cols = [c for c in Epl_data_ready.columns if c.endswith('_ema')]
any_nan = Epl_data_ready[ema_cols].isna().any(axis=1)

#eru nýliðar og önnur lið ekki örugglega einhverntíma nan
debut = Epl_data_ready[any_nan]
print(debut[['Season','GW','Home_Team','Away_Team']].to_string())

     Season  GW                Home_Team          Away_Team
0     17/18   1                  Arsenal          Leicester
1     17/18   1                 Brighton    Manchester City
2     17/18   1                  Watford          Liverpool
3     17/18   1     West Bromwich Albion        Bournemouth
4     17/18   1              Southampton            Swansea
5     17/18   1                  Chelsea            Burnley
6     17/18   1           Crystal Palace       Huddersfield
7     17/18   1                  Everton              Stoke
8     17/18   1         Newcastle United          Tottenham
9     17/18   1        Manchester United           West Ham
381   18/19   1  Wolverhampton Wanderers            Everton
382   18/19   1                   Fulham     Crystal Palace
383   18/19   1              Bournemouth            Cardiff
760   19/20   1                Liverpool            Norwich
761   19/20   1                Tottenham        Aston Villa
763   19/20   1              Bournemouth

In [134]:
#bætum við fyrir líkanið early season flaggi
Epl_data_ready['EarlySeason'] = (Epl_data_ready.groupby('Season')['GW']
                                 .transform(lambda x: x <= 5).astype(int))

In [135]:
#nokkrir hlutir sem þarf að skoða, nan í fyrstu leikjum nýliða og nan í fyrstu umferð 17/18.
ema_cols = [c for c in Epl_data_ready.columns if c.endswith('_ema')]
stats = sorted(set(c.replace('Home_','').replace('Away_','') for c in ema_cols))

#setjum inn 25% mean fyrir nýliða, eðlilegt mat á gæðum þeirra(mjög ólíklegt að nýliðar komi inn eins og stormsveipur)
not_first_season = Epl_data_ready['Season'] != '17/18'
for s in stats:
    hcol, acol = f'Home_{s}', f'Away_{s}'
    proxy = pd.concat([Epl_data_ready[hcol], Epl_data_ready[acol]]).quantile(0.25)
    Epl_data_ready.loc[not_first_season, hcol] = Epl_data_ready.loc[not_first_season, hcol].fillna(proxy)
    Epl_data_ready.loc[not_first_season, acol] = Epl_data_ready.loc[not_first_season, acol].fillna(proxy)

#droppa upphafsleikjum 17/18
before = len(Epl_data_ready)
Epl_data_ready = Epl_data_ready.dropna(subset=ema_cols).reset_index(drop=True)
print(f"dropped {before - len(Epl_data_ready)} rows")

dropped 10 rows


In [136]:
#Þetta er þá tilbúið til að halda áfram í eda og prófanir, loksins.
Epl_data_ready.to_csv('drive/My Drive/Epl_data_ready.csv')
Epl_data_ready = pd.read_csv('drive/My Drive/Epl_data_ready.csv')
Epl_data_ready.head()

,Unnamed: 0,GameId,Date,Home_Team,Away_Team,Home_Goals,Away_Goals,Home_xG,Away_xG,Home_Deep,Away_Deep,Home_ppda,Away_ppda,OddsH,OddsD,OddsA,Matchweek,Home_Possession,Away_Possession,Home_Total_Shots,Away_Total_Shots,Home_Shots_On_Target,Away_Shots_On_Target,Home_Corners,Away_Corners,Home_Saves,Away_Saves,Home_Shots_Off_Target,Away_Shots_Off_Target,Home_Shots_Inside_the_Box,Away_Shots_Inside_the_Box,Home_Shots_Outside_the_Box,Away_Shots_Outside_the_Box,Home_Big_Chances_Created,Away_Big_Chances_Created,Home_Touches,Away_Touches,Home_Touches_in_the_opposition_box,Away_Touches_in_the_opposition_box,Home_Blocks,Away_Blocks,Home_Interceptions,Away_Interceptions,Home_Clearances,Away_Clearances,Home_Duels_Won,Away_Duels_Won,Home_Aerial_Duels_Won,Away_Aerial_Duels_Won,Season,Home_Total_Crosses,Home_Total_Crosses_Completed,Away_Total_Crosses,Away_Total_Crosses_Completed,Home_Total_Passes,Home_Total_Passes_Completed,Away_Total_Passes,Away_Total_Passes_Completed,Home_Long_Passes,Home_Long_Passes_Completed,Away_Long_Passes,Away_Long_Passes_Completed,Home_Through_Balls,Home_Through_Balls_Completed,Away_Through_Balls,Away_Through_Balls_Completed,Home_Tackles_Won_Completed,Away_Tackles_Won_Completed,target,GW,HTP,ATP,HTFormPts,HTForm,ATFormPts,ATForm,Home_Rating,Away_Rating,Home_Goals_ema,Home_xG_ema,Home_Deep_ema,Home_ppda_ema,Home_Possession_ema,Home_Total_Shots_ema,Home_Shots_On_Target_ema,Home_Corners_ema,Home_Saves_ema,Home_Shots_Off_Target_ema,Home_Shots_Inside_the_Box_ema,Home_Shots_Outside_the_Box_ema,Home_Big_Chances_Created_ema,Home_Touches_ema,Home_Touches_in_the_opposition_box_ema,Home_Blocks_ema,Home_Interceptions_ema,Home_Clearances_ema,Home_Duels_Won_ema,Home_Aerial_Duels_Won_ema,Home_Total_Crosses_ema,Home_Total_Crosses_Completed_ema,Home_Total_Passes_ema,Home_Total_Passes_Completed_ema,Home_Long_Passes_ema,Home_Long_Passes_Completed_ema,Home_Through_Balls_ema,Home_Through_Balls_Completed_ema,Home_Tackles_Won_Completed_ema,Away_Goals_ema,Away_xG_ema,Away_Deep_ema,Away_ppda_ema,Away_Possession_ema,Away_Total_Shots_ema,Away_Shots_On_Target_ema,Away_Corners_ema,Away_Saves_ema,Away_Shots_Off_Target_ema,Away_Shots_Inside_the_Box_ema,Away_Shots_Outside_the_Box_ema,Away_Big_Chances_Created_ema,Away_Touches_ema,Away_Touches_in_the_opposition_box_ema,Away_Blocks_ema,Away_Interceptions_ema,Away_Clearances_ema,Away_Duels_Won_ema,Away_Aerial_Duels_Won_ema,Away_Total_Crosses_ema,Away_Total_Crosses_Completed_ema,Away_Total_Passes_ema,Away_Total_Passes_Completed_ema,Away_Long_Passes_ema,Away_Long_Passes_Completed_ema,Away_Through_Balls_ema,Away_Through_Balls_Completed_ema,Away_Tackles_Won_Completed_ema,EarlySeason
0,0,11,2017/08/19,Bournemouth,Watford,0,2,0.766694,2.177170,7,7,10.3125,9.2000,0.500000,0.277778,0.250000,Matchweek 2,0.558,0.442,6.0,19.0,2.0,7.0,8.0,5.0,5.000000,2.00,1.0,7.0,2.000000,3.000000,2.0,6.0,0.0,1.0,617.0,558.0,22.0,34.0,9.0,9.0,8.0,18.0,31.0,28.0,45.0,51.0,16.0,17.0,17/18,15.0,0.27,20.0,0.20,438.0,0.77,338.0,0.78,64.0,0.50,64.0,0.50,0.0,0.0,1.0,1.00,0.64,0.65,L,2,0,1,0.0,L,1.0,D,1337.689105,1500.000000,0.0,0.378659,6.0,12.0000,0.713,9.0,2.0,2.0,5.0,2.0,1.0,4.0,0.0,775.0,19.0,6.0,5.0,17.0,52.0,18.0,24.0,0.21,612.0,0.86,68.0,0.37,1.0,0.0,0.81,3.0,2.176470,2.0,11.2609,0.456,9.0,4.0,3.0,2.0,4.0,0.0,3.0,1.0,595.0,11.0,2.0,11.0,28.0,60.0,22.0,16.0,0.13,395.0,0.70,84.0,0.42,2.0,0.5,0.63,1
1,1,12,2017/08/19,Burnley,West Bromwich Albion,0,1,1.372420,0.653058,4,0,5.5000,14.0000,0.380228,0.312500,0.333333,Matchweek 2,0.678,0.322,20.0,8.0,0.0,1.0,5.0,5.0,3.363636,3.75,15.0,5.0,1.778626,1.166667,9.0,3.0,2.0,0.0,665.0,405.0,20.0,9.0,7.0,8.0,7.0,8.0,33.0,36.0,52.0,44.0,24.0,20.0,17/18,33.0,0.24,13.0,0.31,471.0,0.79,228.0,0.56,87.0,0.46,73.0,0.33,0.0,0.0,0.0,0.00,0.89,0.33,L,2,3,3,3.0,W,3.0,W,1662.310895,1662.310895,3.0,0.564237,3.0,23.7500,0.381,10.0,5.0,5.0,4.0,4.0,2.0,2.0,0.0,485.0,12.0,8.0,13.0,42.0,46.0,22.0,16.0,0.50,320.0,0.75,72.0,0.49,0.0,0.0,0.50,1.0,1.183990,6.0,17.3889,0.287,16.0,6.0,8.0,2.0,9.0,3.0,5.0,1.0,419.0,19.0,6.0,